In [1]:
import pandas as pd
import sqlite3
import numpy as np
import warnings
warnings.filterwarnings('ignore')

db_path = r'C:\Users\beril.oztan\Desktop\rossmann-demand-forecast\data\processed\rossmann.db'
conn = sqlite3.connect(db_path)

def query(sql):
    return pd.read_sql_query(sql, conn)

print("Bağlantı tamam!")

Bağlantı tamam!


In [2]:
from xgboost import XGBRegressor
import pickle

features = [
    'DayOfWeek', 'Promo', 'SchoolHoliday', 'CompetitionDistance',
    'Promo2', 'Year', 'Month', 'Week', 'Quarter', 'DayOfYear',
    'IsWeekend', 'IsDecember', 'lag_7', 'lag_14', 'lag_30',
    'rolling_mean_7', 'rolling_mean_30'
]

# Veriyi yükle
df = query("SELECT * FROM sales_features ORDER BY Store, Date")
df['Date'] = pd.to_datetime(df['Date'])

# Train/test split — modelleme notebook'undaki ile aynı
split_date = '2015-06-01'
train = df[df['Date'] < split_date]
test = df[df['Date'] >= split_date]

X_train = train[features]
y_train = train['Sales']
X_test = test[features]
y_test = test['Sales']

# Modeli tekrar eğit
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)

# Test verisi için tahmin yap
test = test.copy()
test['PredictedSales'] = xgb_model.predict(X_test)

print(f"Tahminler tamamlandı: {len(test)} satır")
test[['Store', 'Date', 'Sales', 'PredictedSales']].head()

Tahminler tamamlandı: 58611 satır


,Store,Date,Sales,PredictedSales
699,1,2015-06-01,5774,7201.183105
700,1,2015-06-02,5450,6212.455078
701,1,2015-06-03,5809,5278.543457
702,1,2015-06-05,5384,5233.102539
703,1,2015-06-06,4183,3926.774902


In [3]:
# Mağaza bazında haftalık tahmin ve stok önerisi hesaplıyoruz

# Güvenlik stoğu formülü:
# safety_stock = Z * std(talep) * sqrt(lead_time)
# Z = 1.65 → %95 servis seviyesi (100 siparişten 95'ini karşıla)
# lead_time = 7 gün (sipariş verdikten 7 gün sonra ürün gelir)

Z = 1.65        # %95 servis seviyesi
lead_time = 7   # gün

# Mağaza bazında haftalık özet
inventory = test.groupby('Store').agg(
    AvgPredictedSales=('PredictedSales', 'mean'),  # Ortalama günlük tahmin
    StdPredictedSales=('PredictedSales', 'std'),   # Talep değişkenliği
    TotalActualSales=('Sales', 'sum'),              # Gerçek toplam satış
    TotalPredictedSales=('PredictedSales', 'sum')   # Tahmin toplam satış
).reset_index()

# Güvenlik stoğu hesapla
inventory['SafetyStock'] = (
    Z * inventory['StdPredictedSales'] * np.sqrt(lead_time)
).round(0)

# Haftalık sipariş önerisi = 7 günlük tahmin + güvenlik stoğu
inventory['WeeklyOrderQty'] = (
    inventory['AvgPredictedSales'] * 7 + inventory['SafetyStock']
).round(0)

print("=== Stok Optimizasyonu ===")
inventory[['Store', 'AvgPredictedSales', 'SafetyStock', 'WeeklyOrderQty']].head(10)

=== Stok Optimizasyonu ===


,Store,AvgPredictedSales,SafetyStock,WeeklyOrderQty
0,1,4519.258789,4212.0,35847.0
1,2,5060.955078,5951.0,41378.0
2,3,7265.330566,7643.0,58500.0
3,4,9859.500977,7528.0,76545.0
4,5,4866.974609,6146.0,40215.0
5,6,4801.534668,5238.0,38849.0
6,7,9409.419922,9411.0,75277.0
7,8,6492.089844,6869.0,52314.0
8,9,7715.046875,6627.0,60632.0
9,10,6076.155273,5179.0,47712.0


In [4]:
# Stok önerilerini veritabanına yazıyoruz
# Dashboard'da kullanacağız

conn2 = sqlite3.connect(db_path)

inventory.to_sql('inventory_recommendations', conn2, if_exists='replace', index=False)

print(f"inventory_recommendations tablosu kaydedildi: {len(inventory)} mağaza")
conn2.close()

inventory.describe()

inventory_recommendations tablosu kaydedildi: 1115 mağaza


,Store,AvgPredictedSales,StdPredictedSales,TotalActualSales,TotalPredictedSales,SafetyStock,WeeklyOrderQty
count,1115.00000,1115.000000,1115.000000,1.115000e+03,1.115000e+03,1115.000000,1115.000000
mean,558.00000,7222.156738,1544.532104,3.764000e+05,3.800496e+05,6742.642090,57297.734375
std,322.01708,2431.944092,396.140594,1.333561e+05,1.314550e+05,1729.367065,18488.806641
min,1.00000,2817.395264,701.004700,1.388850e+05,1.483168e+05,3060.000000,23765.000000
25%,279.50000,5586.145020,1264.573547,2.880815e+05,2.924603e+05,5520.500000,44803.000000
50%,558.00000,6888.104492,1502.568359,3.564440e+05,3.611653e+05,6559.000000,54745.000000
75%,836.50000,8356.321289,1742.579590,4.332735e+05,4.414700e+05,7607.500000,66050.000000
max,1115.00000,21476.560547,4143.766113,1.284037e+06,1.278936e+06,18090.000000,166040.000000
